# Recherche de rendez-vous automatique sur Doctolib 
*Ceci est une expérimentation, et n'est pas encore terminé*

## But initial

1. utiliser [Selenium](https://www.selenium.dev/) et [selenium Python](https://selenium-python.readthedocs.io/installation.html) pour automatiser l'utilisation du site <https://partners.doctolib.fr/hopital-public/perigueux/vaccination-covid?speciality_id=5494&enable_cookies_consent=1>,                                     

2. vérifier si le site a changé, et m'envoyer un SMS si oui.

- Auteur : [Lilian Besson](https://perso.crans.org/besson/) ([@Naereen](https://GitHub.com/Naereen) sur GitHub)
- Licence : [MIT Licensed](https://lbesson.mit-license.org/)
- Date : Mardi 09 février 2021

---
## Premier tutoriel pour prendre en main selenium

Note : je n'avais jamais utilisé selenium ou ce genre de module pour contrôler un navigateur en mode "headless".
Ce n'est pas très compliqué, et j'ai trouvé que ce petit tutoriel est bien fait : https://pythonbasics.org/selenium-firefox-headless/
[Cette documentation est bien complète, pour le module Python de selenium](https://selenium-python.readthedocs.io/).

In [13]:
URL = "https://partners.doctolib.fr/hopital-public/perigueux/vaccination-covid?speciality_id=5494&enable_cookies_consent=1"

In [14]:
from selenium import webdriver

try:
    print(f"Downloading '{URL}'...")
    firefoxOptions = webdriver.FirefoxOptions()
    firefoxOptions.headless = True
    browser = webdriver.Firefox(options=firefoxOptions)

    browser.get(URL)
    print(browser.page_source[:500])
finally:
    try:
        browser.close()
    except:
        pass

<html data-theme="doctolib2023" lang="fr"><head data-country="fr" data-env="production"><meta charset="utf-8"><meta content="width=device-width, initial-scale=1, minimum-scale=1.0" name="viewport"><meta content="noindex" name="robots"><meta content="nofollow" name="robots"><meta content="Trouvez rapidement un spécialiste près de chez vous et prenez rendez-vous gratuitement en ligne en quelques clics" name="description"><script src="https://browser.sentry-cdn.com/6.19.7/bundle.min.js" crossorigin


J'ai bien réussi à installer et utiliser Selenium. Ca marche bien !

## Solution

J'aurai besoin de ces modules là :

In [15]:
import time
from datetime import datetime
import urllib.request
import subprocess
from selenium import webdriver
from selenium.webdriver.common.by import By

Vérifions le contenu de la page, et voir si elle affiche ce message suivant :

> En raison d'une forte demande, ce centre n'a plus de disponibilités : 3362 vaccinations vont avoir lieu dans les 28 prochains jours. Réessayez prochainement ou cherchez un autre centre.

In [16]:
print(f"Downloading '{URL}'...")
firefoxOptions = webdriver.FirefoxOptions()
firefoxOptions.headless = True
browser = webdriver.Firefox(options=firefoxOptions)

browser.get(URL)

In [17]:
message = "ce centre n'a plus de disponibilités"

if message in browser.page_source:
    print(f"{URL} indique :\n{message}")
else:
    print(f"{URL} n'indique pas :\n{message}\nPeut être qu'il y a des disponibilités désormais !")
    !FreeSMS.py f"{URL} semble indiquer qu'il y a des disponibilités désormais."

https://partners.doctolib.fr/hopital-public/perigueux/vaccination-covid?speciality_id=5494&enable_cookies_consent=1 n'indique pas :
ce centre n'a plus de disponibilités
Peut être qu'il y a des disponibilités désormais !


'FreeSMS.py' n'est pas reconnu en tant que commande interne
ou externe, un programme ex�cutable ou un fichier de commandes.


In [18]:
URL = "https://www.doctolib.fr/search?keyword=medecin-generaliste&location=versailles"

In [19]:
firefoxOptions = webdriver.FirefoxOptions()
firefoxOptions.headless = True
browser = webdriver.Firefox(options=firefoxOptions)
browser.get(URL)
contenu = browser.page_source
#print(contenu[:5000])

In [23]:
import time

h2s = browser.find_elements(By.TAG_NAME, "h2")
h2s = [h for h in h2s if h.text.startswith("Dr")]

for h2 in h2s:
    browser.execute_script("arguments[0].scrollIntoView();", h2)
    time.sleep(1)  # attendre le chargement de la dispo
    
    medecin = h2.text
    try:
        card = h2.find_element(By.XPATH, "./ancestor::div[contains(@class,'dl-card')]")
        texte = card.text

        if "Ce soignant réserve la prise de rendez-vous en ligne aux patients déjà suivis" in texte:
            continue
        if "Prochaine disponibilité" in texte:
            dispo = texte.split("Prochaine disponibilité")[-1].strip().replace("\n", " ")
        else:
            dispo = "Aucune disponibilité"
        print(f"Texte de la carte :\n{texte}\n---")     
    except:
        dispo = "Aucune disponibilité"
   
    print(f"{medecin} | {dispo}")

Texte de la carte :
Dr Stéphanie Lescar
Médecin généraliste
16 Rue Saint-Louis
78000 Versailles
Conventionné secteur 1
mardi
24 mars
—
—
—
—
mercredi
25 mars
08:00
08:15
—
—
jeudi
26 mars
—
—
—
—
vendredi
27 mars
—
—
—
—
samedi
28 mars
—
—
—
—
Disponibilités à partir du mardi, mars 24
---
Dr Stéphanie Lescar | Aucune disponibilité
Texte de la carte :
Dr Laurent de Bastard
Médecin généraliste
5 Rue du Bailliage
78000 Versailles
Conventionné secteur 1
mardi
24 mars
—
—
—
—
mercredi
25 mars
—
—
—
—
jeudi
26 mars
—
—
—
—
vendredi
27 mars
—
—
—
—
samedi
28 mars
—
—
—
—
Prochain RDV le 20 avril 2026
Disponibilités à partir du mardi, mars 24
---
Dr Laurent de Bastard | Aucune disponibilité
Texte de la carte :
Dr Camille Julliot de La Morandière
Médecin généraliste
10 Rue André Chénier
78000 Versailles
mardi
24 mars
mercredi
25 mars
jeudi
26 mars
vendredi
27 mars
samedi
28 mars
Aucune disponibilité en ligne
Disponibilités à partir du mardi, mars 24
---
Dr Camille Julliot de La Morandière | Auc

KeyboardInterrupt: 

In [30]:
import time
import re

h2s = browser.find_elements(By.TAG_NAME, "h2")
h2s = [h for h in h2s if h.text.startswith("Dr")]

for h2 in h2s:
    browser.execute_script("arguments[0].scrollIntoView();", h2)
    time.sleep(1)
    
    try:
        text = ""
        card = h2.find_element(By.XPATH, "./ancestor::div[contains(@class,'dl-card')]")

        if "Ce soignant réserve la prise de rendez-vous en ligne aux patients déjà suivis" in card.text:
            continue
        # Cherche une date avec au moins une heure associée
        text = " ".join(card.text.splitlines())
        match_date_heure = re.search(
            r'(lundi|mardi|mercredi|jeudi|vendredi|samedi|dimanche)\s+\d{1,2}\s+\w+(?:\s+\d{4})?\s+(\d{2}:\d{2})',
            text, re.IGNORECASE
        )

        # Fallback : "Prochain RDV le ..."
        match_prochain = re.search(r'Prochain RDV le \d{1,2} \w+ \d{4}', text)

        if match_date_heure:
            dispo = match_date_heure.group().strip()
        elif match_prochain:
            dispo = match_prochain.group().strip()
        else:
            dispo = "Aucune disponibilité"

        dispo_line = dispo.replace("\n", " ")  

        lines = [l.strip() for l in card.text.splitlines() if l.strip()]
        
        nom        = lines[0]                    # Dr Corinne BOYER
        specialite = lines[1]                    # Médecin généraliste
        adresse    = " ".join(lines[2:4])        # 26 BIS Rue Coste 78000 Versailles
        secteur    = lines[4]                    # Conventionné secteur 1
        
        print(f"{nom} | {specialite}")
        print(f"{adresse} | {secteur}")
        print(f"Dispo: {dispo}")
        print("---")
    except Exception as e:
        print(f"Erreur: {e}")

Dr Stéphanie Lescar | Médecin généraliste
16 Rue Saint-Louis 78000 Versailles | Conventionné secteur 1
Dispo: mercredi 25 mars 08:00
---
Dr Laurent de Bastard | Médecin généraliste
5 Rue du Bailliage 78000 Versailles | Conventionné secteur 1
Dispo: Prochain RDV le 20 avril 2026
---
Dr Camille Julliot de La Morandière | Médecin généraliste
10 Rue André Chénier 78000 Versailles | mardi
Dispo: Aucune disponibilité
---
Dr Camille CHAUSSY | Médecin généraliste
17 Passage du Marquis de la Londe 78000 Versailles | Conventionné secteur 1
Dispo: Prochain RDV le 13 avril 2026
---
Dr Mouna Bouzrara | Médecin généraliste
3 Rue de la Porte de Buc 78000 Versailles | Conventionné secteur 1
Dispo: Prochain RDV le 7 avril 2026
---
Dr Hassan Tbahriti | Médecin généraliste
43 Rue d'Anjou 78000 Versailles | Conventionné secteur 1
Dispo: Aucune disponibilité
---
Dr Corinne BOYER | Médecin généraliste
26 BIS Rue Coste 78000 Versailles | Conventionné secteur 1
Dispo: vendredi 27 mars 14:40
---
Dr Dominique F

Quand on a fini, on ferme le navigateur :

In [10]:
browser.close()